<a href="https://colab.research.google.com/github/rozmar/Analyzing-Open-Neuroscience-Data-SZTE-2027/blob/main/Modul%202/02_Electrophysiology.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Electrophysiology

Last week we looked at the **shape** of neurons. Today we look at how they **behave electrically**: the same cells,
the second modality of Patch-seq.

In a patch-clamp recording, a glass pipette is sealed onto the cell membrane and then opened, so that it has
electrical access to the inside of the cell. In **current clamp** we inject a known current through the pipette and
record the membrane voltage. The Allen Institute ran the same set of protocols on every cell; the most useful one is
the **long square**: a 1 second step of constant current, repeated with different amplitudes, negative (the cell
hyperpolarizes) and positive (the cell depolarizes and, above a threshold, fires action potentials).

**Today we will:**
1. look at the electrophysiology features that the Allen Institute computed for every cell,
2. open the raw recording of one cell (an **NWB** file) and plot it,
3. measure the cell ourselves: resting potential, input resistance, spikes, f–I curve, rheobase,
4. compare three cell types,
5. compare all cells, in mouse and in human,
6. and finally connect physiology to anatomy: do bigger neurons behave differently?

**From the bootcamp you will need:** dictionaries and loops, writing functions, `numpy` arrays and boolean masks,
`pandas` (`merge`, `groupby`), and `matplotlib` subplots.

### How to use this notebook

As last week, **eight** code cells contain a small mistake on purpose: 🐞 marks a bug to fix, ✏️ marks something
to fill in (replace `FILL_IN`). Each one is numbered (1/8 … 8/8), and its comment tells you which bootcamp notebook
(in `Modul 1`) and which section to look back at. The fixes are short: one word or one line.

Many of these cells end with a check, `assert ..., "message"`. While something is still wrong, the check stops with
the message; when it is right, the cell runs without complaint. Don't change the checks!


## 0. Setup

As last week: on Colab this downloads the course repository and builds the course data (about a minute). We also
need **pynwb**, the Python package for reading NWB files.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/Analyzing-Open-Neuroscience-Data-SZTE-2027"
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/rozmar/Analyzing-Open-Neuroscience-Data-SZTE-2027.git {REPO}
    os.chdir(REPO + "/Modul 2")
    %pip install -q pynwb

DATA = Path("data")
if not (DATA / "cells.csv").exists():
    print("Building the course data, this takes about a minute...")
    %run 00_Build_Course_Data.ipynb
print("Course data is in:", DATA.resolve())

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import pynwb

# NWB files from 2018-2020 trigger harmless version warnings; hide them
warnings.filterwarnings("ignore", module="hdmf")
warnings.filterwarnings("ignore", module="pynwb")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

COLORS = {"Pvalb": "#D93137", "Sst": "#FF9900", "Sst Chodl": "#B1B10C", "Vip": "#B864CC",
          "Lamp5": "#DA808C", "Sncg": "#6B8E23", "Lamp5/Pax6/Other": "#DA808C"}

## 1. The features table

For every recorded cell, the Allen Institute has already measured dozens of **features** from the recordings:
the resting membrane potential, the input resistance, how much current is needed to fire, how wide the action
potentials are, and so on. They are in `mouse_ephys_features.csv` (and `human_ephys_features.csv`), one row per cell,
joined to `cells.csv` by `cell_id`, just like the morphology features last week.

In [ ]:
cells = pd.read_csv(DATA / "cells.csv", low_memory=False)
cells_by_id = cells.set_index("cell_id")

mouse_ephys = pd.read_csv(DATA / "mouse_ephys_features.csv")
print(mouse_ephys.shape)
mouse_ephys.columns.tolist()

Every column is described in `data/DATA_DICTIONARY.md`; the unit is at the end of the name (`_mv` millivolt,
`_pa` picoampere, `_mohm` megaohm, `_ms` millisecond). The few we will use today:

| column | meaning |
|---|---|
| `vrest_mv` | resting membrane potential |
| `input_resistance_mohm` | how much the voltage changes for a given current (Ohm's law: R = ΔV / ΔI) |
| `threshold_i_long_square_pa` | **rheobase**: the smallest long-square current that makes the cell fire |
| `f_i_curve_slope` | how fast the firing rate grows with more current (Hz per pA) |
| `ap1_width_long_square_ms` | width of the first action potential |
| `adaptation` | how much the spikes slow down during a step (0 = perfectly regular firing) |

Today's example cell is one you already know: `mouse_601506507`, the first cell of `cells.csv` (section 1.3 of
the anatomy lesson). It is a Vip cell from layer 4, with a reconstruction, too.

**Feel free to pick your own cell if you fancy!**

In [ ]:
example_id = "mouse_601506507"
features = mouse_ephys.set_index("cell_id")

print(cells_by_id.loc[example_id, ["subclass", "t_type", "met_type", "layer"]])
features.loc[example_id, ["vrest_mv", "input_resistance_mohm", "threshold_i_long_square_pa",
                          "f_i_curve_slope", "ap1_width_long_square_ms", "adaptation"]]

## 2. One recording

### 2.1 The NWB file

The raw recordings are stored in **NWB** (Neurodata Without Borders) files, a standard format for neuroscience data,
and shared on the [DANDI Archive](https://dandiarchive.org) (mouse: dandiset 000020, human: 000636). The column
`nwb_url` of `cells.csv` is the download link of each cell's file.

**If DANDI is down**, set `USE_BACKUP_NWB = True`: then we use a copy of the example cell's file that the Allen
Institute ships as example data with its [IPFX](https://github.com/AllenInstitute/ipfx) software on GitHub.

In [ ]:
USE_BACKUP_NWB = False

BACKUP_NWB_URL = ("https://media.githubusercontent.com/media/AllenInstitute/ipfx/"
                  "c607a36e25618a6bb8ba5438c1cd945fb429e659/tests/data/nwb/Vip-IRES-Cre%3BAi14-331294.04.01.01.nwb")

NWB_DIR = DATA / "nwb"
NWB_DIR.mkdir(exist_ok=True)

def download_nwb(cell_id):
    """Download the NWB file of a cell (once) and return the path of the local copy."""
    path = NWB_DIR / f"{cell_id}.nwb"
    if not path.exists():
        url = cells_by_id.loc[cell_id, "nwb_url"]
        if USE_BACKUP_NWB and cell_id == "mouse_601506507":
            url = BACKUP_NWB_URL
        if pd.isna(url):
            raise ValueError(f"no NWB link for {cell_id}")
        print("downloading", cell_id, "...")
        response = requests.get(url, timeout=300)
        response.raise_for_status()
        path.write_bytes(response.content)
    return path

def open_nwb(cell_id):
    """Download (once) and open the NWB file of a cell."""
    return pynwb.NWBHDF5IO(str(download_nwb(cell_id)), "r", load_namespaces=True).read()

nwb = open_nwb(example_id)
print(len(nwb.acquisition), "recorded sweeps")
print(list(nwb.acquisition.keys())[:5])

The recorded voltage traces are in `nwb.acquisition`, the injected currents in `nwb.stimulus`. Both behave like
**dictionaries**: one entry per **sweep** (one run of a protocol). The current injected during sweep
`data_00029_AD0` has the same name, ending in `DA0` instead of `AD0`.

A sweep is stored as integers plus a **conversion** factor: `data × conversion` gives the value in SI units
(volts, amperes). We convert to the units electrophysiologists use, millivolts and picoamperes. The time axis
comes from the sampling **rate** (samples per second): sample *k* was taken at time *k / rate*.

In [ ]:
def get_sweep(nwb, name):
    """Time (s), membrane voltage (mV) and injected current (pA) of one sweep."""
    response = nwb.acquisition[name]
    stimulus = nwb.stimulus[name.replace("AD0", "DA0")]
    # 🐞 FIX ME 1/8: the voltage comes out in the wrong unit. 1 V = how many mV?
    # Bootcamp: 04_Introduction_To_Numpy, "Arithmetic with arrays"
    v = response.data[:] * response.conversion * FILL_IN         # V -> mV
    i = stimulus.data[:] * stimulus.conversion * 1e12         # A -> pA
    t = np.arange(len(v)) / response.rate                     # s
    return t, v, i

name = "data_00040_AD0"                 # one of this cell's sweeps (we will see below what it is)
t, v, i = get_sweep(nwb, name)
assert -150 < v.min() < -20, "a resting neuron sits at about -70 mV: check the units"

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
axes[0].plot(t, v, color="black", linewidth=0.5)
axes[0].set_ylabel("membrane voltage (mV)")
axes[1].plot(t, i, color="tab:blue")
axes[1].set_ylabel("current (pA)")
axes[1].set_xlabel("time (s)")
axes[0].set_title(f"{example_id}, sweep {name}")
plt.show()

**Look at the plot.** The small blip at the very beginning is a *test pulse*, used to check the recording quality.
Then comes the real stimulus: a **step** of constant current, here lasting 1 s. At the start of the step the cell
fires a burst of action potentials, then stops while the voltage stays high: at this strong current the sodium
channels cannot recover (*depolarization block*).

### 2.2 What is in each sweep?

Each sweep knows the name of its protocol (`stimulus_description`), but the names are codes, and the Allen Institute
changed them over the years. Cells recorded before 2018 use names like `C1LSCOARSE...` (C1 = core protocol 1,
LS = long square); later cells use `X1PS_SubThresh`, `X3LP_Rheo`, `X4PS_SupraThresh`, ... Instead of learning the
codes, we read what each sweep **actually did** from its injected current.

The step is where the current is switched on, after the test pulse (which is always in the first 0.1 s):

In [ ]:
def find_step(t, i):
    """Start (s), end (s) and current (pA) of the step in a sweep."""
    on = (np.abs(i) > 1) & (t > 0.1)          # current is on, and we are past the test pulse
    if not on.any():
        return np.nan, np.nan, 0.0
    start, end = t[on][0], t[on][-1]
    current = np.median(i[(t >= start) & (t <= end)])
    return start, end, current

find_step(t, i)

Now we run this on every sweep and collect the results in a table, with a loop. We skip the *voltage-clamp*
sweeps (the quality checks at the start and end of the experiment): they inject voltage, not current.

One more thing, because real data are messy: sometimes the experimenter stopped a sweep early (for example
because the cell became unstable). The rest of that sweep is then filled with exactly 0 mV, which a living cell
never shows. We mark these sweeps as incomplete.

In [ ]:
def sweep_table(nwb):
    """One row per current-clamp sweep: protocol, step timing and current, and whether it was fully recorded."""
    rows = []
    for name, series in nwb.acquisition.items():
        if not isinstance(series, pynwb.icephys.CurrentClampSeries):
            continue
        t, v, i = get_sweep(nwb, name)
        start, end, current = find_step(t, i)
        last_100ms = v[(t >= end - 0.1) & (t < end)]
        rows.append({"sweep": name,
                     "protocol": series.stimulus_description,
                     "step_start_s": start,
                     "step_duration_s": end - start,
                     "current_pa": current,
                     "complete": not (last_100ms == 0).all()})
        # 🐞 FIX ME 2/8: the table has only one row. Which lines belong to the loop, and which come after it?
        # Bootcamp: 02_Basic_Python_II_Control_Flow_and_Functions, "for-loops" and "Functions"
        return pd.DataFrame(rows)

sweeps = sweep_table(nwb)
assert len(sweeps) > 10, "the table should have one row for every current-clamp sweep"
sweeps

The **long squares** are the sweeps with a 1 s step. (The others are 3 ms *short squares* and slow *ramps*.)
We keep only the completely recorded ones, in the order they were recorded.

The long squares were recorded in **series**: one sweep after the other with the same protocol, the current
increasing (or decreasing) step by step. A new series starts where the protocol changes from one sweep to the
next: `.shift()` moves the column down by one row, so we can compare each sweep with the one before it, and
`.cumsum()` counts the changes.

In [ ]:
is_long_square = (sweeps["step_duration_s"] > 0.95) & (sweeps["step_duration_s"] < 1.05)
# ✏️ FILL IN 3/8: we want the long squares that were recorded completely. Which column says that?
# Bootcamp: 07_Introduction_To_Pandas, "Return a subset of data given a specific threshold or condition"
long_squares = sweeps[is_long_square & sweeps[FILL_IN]].copy()

new_series = long_squares["protocol"] != long_squares["protocol"].shift()
long_squares["series"] = new_series.cumsum()
long_squares

## 3. Measuring the cell ourselves

### 3.1 Resting potential

Before the step, no current is injected: the voltage there is the **resting membrane potential**. We take the mean
of the 100 ms just before the step of the sweep we plotted.

In [ ]:
start, end, current = find_step(t, i)
before = (t >= start - 0.1) & (t < start)
v_rest = v[before].mean()
print("resting potential: %.1f mV   (Allen: %.1f mV)" % (v_rest, features.loc[example_id, "vrest_mv"]))

### 3.2 Input resistance

When we inject a negative current, the voltage drops. By **Ohm's law** (*V = I · R*), the size of the drop tells us
the cell's **input resistance**: a cell with high resistance changes its voltage a lot for a little current.
Let's look at the hyperpolarizing (negative) steps, one series per column: the voltage on top, the injected
current below, in the same color.

In [ ]:
negative = long_squares[long_squares["current_pa"] < 0]
series_ids = negative["series"].unique()

fig, axes = plt.subplots(2, len(series_ids), figsize=(5 * len(series_ids), 5), sharex=True, sharey="row",
                         squeeze=False, gridspec_kw={"height_ratios": [3, 1]})
rows = []
for col, series_id in enumerate(series_ids):
    ax_v, ax_i = axes[0, col], axes[1, col]
    one_series = negative[negative["series"] == series_id]
    for k, (_, sweep) in enumerate(one_series.iterrows()):
        t, v, i = get_sweep(nwb, sweep["sweep"])
        start, end, current = find_step(t, i)
        baseline = v[(t >= start - 0.1) & (t < start)].mean()
        # 🐞 FIX ME 4/8: the voltage DROPS during these steps. Is its lowest point the max or the min?
        # Bootcamp: 04_Introduction_To_Numpy, "Reduction methods: min(), max(), mean(), and std()"
        lowest = v[(t >= start) & (t <= end)].max()          # the largest drop during the step
        rows.append({"series": series_id, "current_pa": current, "delta_v_mv": lowest - baseline})
        color = f"C{k}"                                        # matplotlib's k-th default color
        ax_v.plot(t - start, v, color=color, linewidth=0.7, label=f"{current:.0f} pA")
        ax_i.plot(t - start, i, color=color)
    ax_v.set_title(f"series {series_id}: {one_series['protocol'].iloc[0]}")
    ax_i.set_xlabel("time from step start (s)")
    ax_v.legend(fontsize=7, loc="upper right", ncol=3)
axes[0, 0].set_ylabel("membrane voltage (mV)")
axes[1, 0].set_ylabel("current (pA)")
axes[0, 0].set_xlim(-0.1, 1.2)
plt.tight_layout()
plt.show()

hyper = pd.DataFrame(rows)
hyper

In many cells the voltage first drops fast and then creeps back up a little during the step. This is the **sag**,
caused by a special channel (HCN, *I*<sub>h</sub>) that opens when the cell hyperpolarizes. This Vip cell has very
little of it, but that is why we measure the **largest** drop, at the bottom of the curve.

The voltage change grows in proportion to the current, so the points lie on a line. The **slope** of that line is
the resistance. `np.polyfit(x, y, 1)` fits a straight line and returns its slope and intercept. The slope is in
mV / pA, which is GΩ (gigaohm); times 1000 gives MΩ.

In [ ]:
slope, intercept = np.polyfit(hyper["current_pa"], hyper["delta_v_mv"], 1)
r_in = slope * 1000
allen_r_in = features.loc[example_id, "input_resistance_mohm"]
assert 0.5 < r_in / allen_r_in < 2, "far from Allen's value: look at how the voltage drop was measured (bug 4/8)"
print("input resistance: %.0f MOhm   (Allen: %.0f MOhm)" % (r_in, features.loc[example_id, "input_resistance_mohm"]))

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(hyper["current_pa"], hyper["delta_v_mv"])
x = np.array([hyper["current_pa"].min(), 0])
ax.plot(x, slope * x + intercept, "--", color="gray")
ax.set_xlabel("injected current (pA)")
ax.set_ylabel("voltage change (mV)")
plt.show()

### 3.3 Counting spikes

An action potential is easy to find: the voltage shoots above 0 mV, which never happens otherwise. A spike
**starts** where the voltage was below 0 mV in one sample and above 0 mV in the next. With numpy this is a
comparison of the trace with itself, shifted by one sample:

In [ ]:
def count_spikes(t, v, start, end):
    """Number of action potentials between start and end (upward crossings of 0 mV)."""
    step = v[(t >= start) & (t <= end)]
    above = step > 0
    # 🐞 FIX ME 5/8: this counts every sample above 0 mV, not the moments the voltage crosses 0 mV.
    # The comment says "below 0 mV before": which operator turns True into False?
    # Bootcamp: 04_Introduction_To_Numpy, "Boolean operations"
    crossings = above[1:] & above[:-1]        # below 0 mV before, above 0 mV now
    return crossings.sum()

name = long_squares.loc[long_squares["current_pa"].idxmax(), "sweep"]      # the sweep with the largest current
t, v, i = get_sweep(nwb, name)
start, end, current = find_step(t, i)
n_spikes = count_spikes(t, v, start, end)
print(n_spikes, "spikes at", round(current), "pA")
assert n_spikes < 50, "far too many spikes for this cell: are we counting crossings, or samples above 0 mV?"

### 3.4 The f–I curve and the rheobase

Now we do this for every long square: how many spikes per second (*f*, firing rate) for how much current (*I*)?
Some currents were repeated, so we average them with `groupby`. The **rheobase** is the smallest current that made
the cell fire.

In [ ]:
def fi_curve(nwb, long_squares):
    """Firing rate (Hz) for each current (pA) of the long squares."""
    rows = []
    for _, sweep in long_squares.iterrows():
        t, v, i = get_sweep(nwb, sweep["sweep"])
        start, end, current = find_step(t, i)
        rows.append({"current_pa": round(current),
                     "rate_hz": count_spikes(t, v, start, end) / (end - start)})
    return pd.DataFrame(rows).groupby("current_pa")["rate_hz"].mean()

fi = fi_curve(nwb, long_squares)
# 🐞 FIX ME 6/8: the rheobase is the smallest current that made the cell FIRE. Which rows do we select here?
# Bootcamp: 07_Introduction_To_Pandas, "Select data given a specific threshold or condition"
rheobase = fi[fi == 0].index.min()
assert rheobase > 0, "a negative current cannot make the cell fire"
print("rheobase: %.0f pA   (Allen: %.0f pA)" % (rheobase, features.loc[example_id, "threshold_i_long_square_pa"]))

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fi.index, fi.values, "o-", color=COLORS["Vip"])
ax.axvline(rheobase, linestyle="--", color="gray")
ax.set_xlabel("injected current (pA)")
ax.set_ylabel("firing rate (Hz)")
ax.set_title("f–I curve of " + example_id)
plt.show()

**Compare with Allen.** Our numbers should be close to the precomputed ones, but not always identical: the Allen
pipeline checks the quality of every sweep and uses more careful definitions. Checking your own result against an
independent calculation is the habit that matters.

## 4. Three cell types

With the functions above, one cell takes three lines. Let's run them on the Pvalb, Sst and Vip cells from last
week's gallery and look at the sweep with the most spikes of each, and at their f–I curves.

In [ ]:
# 🐞 FIX ME 7/8: KeyError! Look at the cell_id in the error message: two ids got glued together.
# Bootcamp: 01_Basic_Python_I_Object_and_Data_Structures, the part on lists
gallery = ["mouse_606271263", "mouse_740135032" "mouse_694146546"]      # Pvalb, Sst, Vip (anatomy lesson)

fig, axes = plt.subplots(len(gallery), 2, figsize=(11, 2.6 * len(gallery)),
                         gridspec_kw={"width_ratios": [3, 1]})
for row, cell_id in zip(axes, gallery):
    sc = cells_by_id.loc[cell_id, "subclass"]
    nwb_c = open_nwb(cell_id)
    sweeps_c = sweep_table(nwb_c)
    ls_c = sweeps_c[(sweeps_c["step_duration_s"] > 0.95) & (sweeps_c["step_duration_s"] < 1.05) & sweeps_c["complete"]]
    fi_c = fi_curve(nwb_c, ls_c)
    best_current = fi_c.idxmax()                                   # the current with the highest rate
    best = ls_c.loc[ls_c["current_pa"].round() == best_current, "sweep"].iloc[0]
    t, v, i = get_sweep(nwb_c, best)
    start, end, current = find_step(t, i)
    row[0].plot(t - start, v, color=COLORS[sc], linewidth=0.5)
    row[0].set_xlim(-0.1, 1.2)
    row[0].set_ylabel(f"{sc}\n(mV)")
    row[1].plot(fi_c.index, fi_c.values, "o-", color=COLORS[sc])
    row[1].set_ylabel("rate (Hz)")
axes[-1][0].set_xlabel("time from step start (s)")
axes[-1][1].set_xlabel("current (pA)")
plt.tight_layout()
plt.show()

**Look at the traces.** Pvalb cells are the classic **fast-spiking** interneurons: narrow spikes at high, regular
rates. Sst cells often fire more slowly and **adapt** (the spikes get further apart during the step). How does the Vip
cell compare? And the f–I curves: which cell's rate grows fastest with current?

## 5. All cells: the precomputed features

Now the features of all recorded cells. We add the subclass from `cells.csv` with a `merge`, and reuse last week's
strip plot.

In [ ]:
mouse_ephys = mouse_ephys.merge(cells[["cell_id", "subclass", "subclass_harmonized"]], on="cell_id")

def strip_plot(df, feature, group, order, ax, offset=0.0, marker="o"):
    """One column of dots per group, with the median as a black line."""
    rng = np.random.default_rng(0)
    for k, g in enumerate(order):
        values = df.loc[df[group] == g, feature].dropna()
        x = k + offset + rng.uniform(-0.15, 0.15, len(values))
        ax.scatter(x, values, s=5, color=COLORS[g], alpha=0.4, marker=marker)
        ax.plot([k + offset - 0.2, k + offset + 0.2], [values.median()] * 2, color="black")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=30)
    ax.set_ylabel(feature)

subclasses = ["Lamp5", "Sncg", "Vip", "Sst", "Sst Chodl", "Pvalb"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feature in zip(axes, ["ap1_width_long_square_ms", "f_i_curve_slope", "adaptation"]):
    strip_plot(mouse_ephys, feature, "subclass", subclasses, ax)
plt.tight_layout()
plt.show()

**Questions.** Which subclass has the narrowest action potentials? The steepest f–I curve? Which one has the most
adaptation, and which ones fire regularly? Does it match what you saw in section 4?

### 5.1 Mouse and human

The human table uses slightly different names, but has the input resistance in MΩ too. (Many human cells were kept
in slice culture for a few days before recording; see the `preparation` column of `cells.csv`.)

In [ ]:
human_ephys = pd.read_csv(DATA / "human_ephys_features.csv")
human_ephys = human_ephys.merge(cells[["cell_id", "subclass_harmonized", "preparation"]], on="cell_id")
human_ephys["input_resistance_mohm"] = human_ephys["input_resistance"]

harmonized = ["Lamp5/Pax6/Other", "Vip", "Sst", "Pvalb"]
fig, ax = plt.subplots(figsize=(7, 4))
strip_plot(mouse_ephys, "input_resistance_mohm", "subclass_harmonized", harmonized, ax, offset=-0.2)
strip_plot(human_ephys, "input_resistance_mohm", "subclass_harmonized", harmonized, ax, offset=+0.2, marker="^")
ax.set_title("input resistance (left: mouse ●, right: human ▲)")
plt.show()

## 6. Anatomy meets physiology

Every bit of cell membrane has ion channels that let a little current leak out. A **bigger** neuron has more
membrane, so more leak, so a **lower input resistance**: the same injected current spreads over more membrane and
changes the voltage less. (This is the basic idea of *cable theory*.)

Is that true for real neurons? For the cells that were both recorded *and* reconstructed, we can merge the
electrophysiology and morphology tables on `cell_id` and check. As "size" we take the total length of the
dendrites and the axon.

In [ ]:
mouse_morph = pd.read_csv(DATA / "mouse_morph_features.csv")
mouse_both = mouse_ephys.merge(mouse_morph, on="cell_id")
mouse_both["total_length_mm"] = mouse_both["basal_dendrite_total_length_mm"] + mouse_both["axon_total_length_mm"]

human_morph = pd.read_csv(DATA / "human_morph_features.csv")
human_both = human_ephys.merge(human_morph, on="cell_id")
# 🐞 FIX ME 8/8: the human lengths are in µm (see last week, section 7.1), but the column name says mm
# Bootcamp: 04_Introduction_To_Numpy, "Arithmetic with arrays"
human_both["total_length_mm"] = (human_both["basal_dendrite_total_length"] + human_both["axon_total_length"])
assert human_both["total_length_mm"].median() < 100, "a typical interneuron is not 100 mm long: check the units"

print(len(mouse_both), "mouse cells and", len(human_both), "human cells with both")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, df, title in [(axes[0], mouse_both, "mouse"), (axes[1], human_both, "human")]:
    for sc in harmonized:
        part = df[df["subclass_harmonized"] == sc]
        ax.scatter(part["total_length_mm"], part["input_resistance_mohm"], s=12,
                   color=COLORS[sc], alpha=0.7, label=sc)
    r = df["total_length_mm"].corr(df["input_resistance_mohm"], method="spearman")
    ax.set_title(f"{title}: r = {r:.2f}")
    ax.set_xlabel("total dendrite + axon length (mm)")
axes[0].set_ylabel("input resistance (MΩ)")
axes[1].legend()
plt.show()

`r` is the **correlation**: −1 means a perfect "the bigger, the lower", 0 means no relation. (We use the *Spearman*
correlation, which only looks at the order of the values, so a few extreme cells cannot dominate it.)

Is this just because some subclasses are bigger *and* have lower resistance? Then the relation would disappear
**within** each subclass. `groupby` lets us compute the correlation for each subclass separately:

In [ ]:
def spearman(df):
    return df["total_length_mm"].corr(df["input_resistance_mohm"], method="spearman")

within = pd.DataFrame({
    "mouse": mouse_both.groupby("subclass_harmonized")[["total_length_mm", "input_resistance_mohm"]].apply(spearman),
    "human": human_both.groupby("subclass_harmonized")[["total_length_mm", "input_resistance_mohm"]].apply(spearman),
})
within.round(2)

**Questions to think about.**
* Does the relation hold in both species? Within each subclass?
* The correlation is clear but far from −1. What else, besides size, could set a neuron's input resistance?
  (Think of ion channels: next lesson is about genes.)
* The axons were cut when the slice was made. How could that affect this analysis?

## Summary

* The Allen Institute recorded every cell with the same protocols; the raw data are NWB files on DANDI.
* From the long-square steps we measured the resting potential, the input resistance (Ohm's law), the spikes, the f–I
  curve and the rheobase, and checked them against the precomputed features.
* Subclasses differ in how they fire: Pvalb cells are fast-spiking, with narrow spikes.
* Anatomy and physiology are linked: bigger neurons tend to have a lower input resistance, in mouse and in human,
  but size is only part of the story.

**Next lesson: transcriptomics.** Which genes make these cells different?